# AutoLyap: Proximal gradient method

[Open in Colab](https://colab.research.google.com/github/ManuUpadhyaya/tutorial_ELLIIT_2026/blob/main/notebooks/proximal_gradient_method.ipynb)

This notebook uses AutoLyap to analyze the proximal gradient method with an iteration-independent Lyapunov certificate.


## Setup

Install AutoLyap. The notebook defaults to
`backend="cvxpy", cvxpy_solver="CLARABEL"`, which works in Google Colab without a commercial solver license.


In [ ]:
%pip install -q autolyap

## Problem setup

Consider the composite minimization problem

$$
\underset{y \in \mathcal{H}}{\operatorname{minimize}}\; f_1(y) + f_2(y),
$$

where $\mathcal{H}$ is a real Hilbert space. The associated inclusion problem is

$$
\text{find } y \in \mathcal{H}\;\text{ such that }\; 0 \in \nabla f_1(y) + \partial f_2(y).
$$

To obtain a linear-convergence certificate, we use the strongly convex specialization
$f_1 \in \mathcal{F}_{\mu,L}(\mathcal{H})$ with $0<\mu<L<+\infty$, and
$f_2 \in \mathcal{F}_{0,\infty}(\mathcal{H})$. Thus, $f_1$ is proper,
lower semicontinuous, $\mu$-strongly convex, and $L$-smooth, while $f_2$ is
proper, lower semicontinuous, and convex.

For an initial point $x^0 \in \mathcal{H}$ and step size $0<\gamma<2/L$, the
proximal gradient update is

$$
(\forall k \in \mathbb{N})\quad
x^{k+1}=\operatorname{prox}_{\gamma f_2}\left(x^k-\gamma\nabla f_1(x^k)\right).
$$

We use AutoLyap to search for a contraction factor $\rho\in[0,1)$ such that

$$
\|x^k-x^\star\|^2 \in \mathcal{O}(\rho^k)\quad\text{as}\quad k\to\infty,
$$

where $x^\star\in\operatorname{Argmin}_{x\in\mathcal{H}} f_1(x)+f_2(x)$.


## State-space representation

$$
\begin{aligned}
\mathbf{x}^{k+1}
&=\left(\begin{bmatrix}\star\end{bmatrix}\otimes I\right)\mathbf{x}^k
+\left(\begin{bmatrix}\star&\star\end{bmatrix}\otimes I\right)\mathbf{u}^k,\\
\mathbf{y}^{k}
&=\left(\begin{bmatrix}\star\\\star\end{bmatrix}\otimes I\right)\mathbf{x}^k
+\left(\begin{bmatrix}\star&\star\\\star&\star\end{bmatrix}\otimes I\right)\mathbf{u}^k,\\
\mathbf{u}^k
&\in \partial f_1(y_1^k)\times\partial f_2(y_2^k).
\end{aligned}
$$

where

$$
\begin{aligned}
\mathbf{x}^k &= x^k,\\
\mathbf{u}^k &= (u_1^k,u_2^k)
=\left(\nabla f_1(x^k),\frac{x^k-x^{k+1}}{\gamma}-\nabla f_1(x^k)\right),\\
\mathbf{y}^k &= (y_1^k,y_2^k)=(x^k,x^{k+1}).
\end{aligned}
$$

To find the entries, use the proximal optimality condition and match

$$
x^{k+1}=x^k-\gamma u_1^k-\gamma u_2^k,
\qquad
 y_1^k=x^k,
\qquad
 y_2^k=x^{k+1}
$$

against the state-space equations above.

The custom class below is intentionally incomplete: replace each starred entry in `A`, `B`, `C`, and `D` before running the Lyapunov search.


In [ ]:
import numpy as np

from autolyap import SolverOptions
from autolyap.algorithms import Algorithm
from autolyap.iteration_independent import IterationIndependent
from autolyap.problemclass import Convex, InclusionProblem, SmoothStronglyConvex


class ProximalGradientMethod(Algorithm):
    def __init__(self, gamma: float) -> None:
        super().__init__(n=1, m=2, m_bar_is=[1, 1], I_func=[1, 2], I_op=[])
        self.gamma = gamma

    def set_gamma(self, gamma: float) -> None:
        self.gamma = gamma

    def get_ABCD(self, k: int):
        gamma = self.gamma

        # Exercise: replace each star in the displayed state-space matrices.
        # Keep the shapes fixed: A is 1x1, B is 1x2, C is 2x1, D is 2x2.
        A = np.array([[...]])
        B = np.array([[..., ...]])
        C = np.array([[...], [...]])
        D = np.array([[..., ...], [..., ...]])

        return A, B, C, D


## Validate the state-space matrices

Run this cell after filling in `A`, `B`, `C`, and `D`. It checks the matrix shapes, checks that all entries are numeric, and tests the equations above on a few sample values before the notebook starts solving SDPs.


In [ ]:
def as_float_matrix(name, value, shape):
    try:
        array = np.asarray(value, dtype=float)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            f"{name} still contains placeholders or non-numeric entries."
        ) from exc
    if array.shape != shape:
        raise ValueError(f"{name} has shape {array.shape}, expected {shape}.")
    if not np.all(np.isfinite(array)):
        raise ValueError(f"{name} contains non-finite entries.")
    return array

validation_gamma = 0.2
validation_algorithm = ProximalGradientMethod(gamma=validation_gamma)
A_test, B_test, C_test, D_test = validation_algorithm.get_ABCD(k=0)
A_test = as_float_matrix("A", A_test, (1, 1))
B_test = as_float_matrix("B", B_test, (1, 2))
C_test = as_float_matrix("C", C_test, (2, 1))
D_test = as_float_matrix("D", D_test, (2, 2))

samples = [
    (1.0, -2.0, 0.5),
    (-0.3, 0.7, -1.1),
    (2.2, 0.0, 1.4),
]
for x_value, u1_value, u2_value in samples:
    x_vec = np.array([x_value])
    u_vec = np.array([u1_value, u2_value])
    expected_next = x_value - validation_gamma * u1_value - validation_gamma * u2_value
    x_next = (A_test @ x_vec + B_test @ u_vec).item()
    y_vec = C_test @ x_vec + D_test @ u_vec
    np.testing.assert_allclose(x_next, expected_next, atol=1e-12)
    np.testing.assert_allclose(y_vec, [x_value, expected_next], atol=1e-12)

print("State-space matrices passed the quick validation.")


## Define problem data and create instances

Choose the problem parameters and create an instance of the inclusion problem and proximal gradient method.


In [ ]:
mu, L, gamma = 1.0, 4.0, 0.2

problem = InclusionProblem([
    SmoothStronglyConvex(mu, L),  # f_1
    Convex(),                     # f_2
])
algorithm = ProximalGradientMethod(gamma=gamma)

solver_options = SolverOptions(backend="cvxpy", cvxpy_solver="CLARABEL")


## Iteration-independent Lyapunov analysis

For iteration-independent analyses, we restrict attention to methods with

$$
(A_k,B_k,C_k,D_k)=(A,B,C,D).
$$

The slides search for Lyapunov inequalities of the form

$$
\mathcal{V}(k+1)\leq \rho\mathcal{V}(k)-\mathcal{R}(k),
\qquad \rho\in[0,1],
$$

where

$$
\mathcal{V}(k)\geq 0 \quad\text{(Lyapunov function)},
\qquad
\mathcal{R}(k)\geq 0 \quad\text{(residual function)}.
$$

AutoLyap uses quadratic ansatzes

$$
\begin{aligned}
\mathcal{V}(k)
&=\left\langle \boldsymbol{\xi}^{k}_{\mathrm{vec}},
(Q\otimes I)\boldsymbol{\xi}^{k}_{\mathrm{vec}}\right\rangle
+q^{\top}\boldsymbol{\xi}^{k}_{\mathrm{func}},\\
\mathcal{R}(k)
&=\left\langle \boldsymbol{\xi}^{k}_{\mathrm{vec}},
(S\otimes I)\boldsymbol{\xi}^{k}_{\mathrm{vec}}\right\rangle
+s^{\top}\boldsymbol{\xi}^{k}_{\mathrm{func}},
\end{aligned}
$$

where $Q,S$ and $q,s$ are the unknown Lyapunov and residual parameters.


## Choose the lower bounds $P,p,T,t$

The zero choice $Q=S=0$ and $q=s=0$ always gives a valid but useless Lyapunov
inequality. To avoid this, the slides prescribe nonnegative lower bounds:

$$
\begin{aligned}
\textbf{C1}\quad
&\mathcal{V}(k+1)\leq \rho\mathcal{V}(k)-\mathcal{R}(k),\\
\textbf{C2}\quad
&\mathcal{V}(k)\geq
\left\langle \boldsymbol{\xi}^{k}_{\mathrm{vec}},
(P\otimes I)\boldsymbol{\xi}^{k}_{\mathrm{vec}}\right\rangle
+p^{\top}\boldsymbol{\xi}^{k}_{\mathrm{func}}
\geq 0,\\
\textbf{C3}\quad
&\mathcal{R}(k)\geq
\left\langle \boldsymbol{\xi}^{k}_{\mathrm{vec}},
(T\otimes I)\boldsymbol{\xi}^{k}_{\mathrm{vec}}\right\rangle
+t^{\top}\boldsymbol{\xi}^{k}_{\mathrm{func}}
\geq 0.
\end{aligned}
$$

For $\rho\in[0,1)$, **C1** and **C3** imply
$\mathcal{V}(k)\leq \rho^k\mathcal{V}(0)$; hence, by **C2**,

$$
0\leq
\left\langle \boldsymbol{\xi}^{k}_{\mathrm{vec}},
(P\otimes I)\boldsymbol{\xi}^{k}_{\mathrm{vec}}\right\rangle
+p^{\top}\boldsymbol{\xi}^{k}_{\mathrm{func}}
\leq \rho^k\mathcal{V}(0)\to 0.
$$

Because $y_1^k=x^k$ in the proximal-gradient state-space representation, we use
`i=1, j=1`. The helper chooses $P,p,T,t$ so that

$$
\begin{aligned}
\left\langle \boldsymbol{\xi}^{k}_{\mathrm{vec}},
(P\otimes I)\boldsymbol{\xi}^{k}_{\mathrm{vec}}\right\rangle
+p^{\top}\boldsymbol{\xi}^{k}_{\mathrm{func}}
&=\|y_1^k-y^\star\|^2=\|x^k-y^\star\|^2,\\
\left\langle \boldsymbol{\xi}^{k}_{\mathrm{vec}},
(T\otimes I)\boldsymbol{\xi}^{k}_{\mathrm{vec}}\right\rangle
+t^{\top}\boldsymbol{\xi}^{k}_{\mathrm{func}}
&=0.
\end{aligned}
$$


In [ ]:
P, p, T, t = (
    IterationIndependent.LinearConvergence.get_parameters_distance_to_solution(
        algorithm,
        i=1,
        j=1,
    )
)


## Do the Lyapunov search

With $P,p,T,t$ fixed, AutoLyap searches for $Q,q,S,s$ satisfying **C1**, **C2**,
and **C3** for the algorithm-consistent points, solutions, and functions in the
chosen problem class. The bisection helper searches over $\rho$ to find the
smallest feasible contraction factor.

In this notebook we set `S_equals_T=True` and `s_equals_t=True`, so
$\mathcal{R}(k)$ is parameterized by the prescribed residual lower bound. Since
that lower bound is zero here, `remove_C3=True` removes the redundant **C3**
constraint from the SDP.


In [ ]:
search_result = IterationIndependent.LinearConvergence.bisection_search_rho(
    problem,
    algorithm,
    P,
    T,
    p=p,
    t=t,
    S_equals_T=True,
    s_equals_t=True,
    remove_C3=True,
    lower_bound=0.0,
    upper_bound=1.0,
    tol=1e-5,
    solver_options=solver_options,
    verbosity=0,
)


## Compare with a standard contraction bound

A standard comparison bound for proximal gradient under these assumptions is

$$
\|x^k-x^\star\|^2 \in \mathcal{O}(\rho^k),
$$

with

$$
\rho = \max\{|1-\gamma L|,\ |1-\gamma\mu|\}^2.
$$


In [ ]:
if search_result["status"] != "feasible":
    raise RuntimeError(
        "No feasible Lyapunov certificate in the requested rho interval."
    )

rho_theory = max(abs(1.0 - gamma * L), abs(1.0 - gamma * mu)) ** 2
rho_autolyap = float(search_result["rho"])
rho_error = abs(rho_autolyap - rho_theory)
rho_tolerance = 2e-4

print(f"status: {search_result['status']}")
print(f"solve_status: {search_result['solve_status']}")
print(f"rho (AutoLyap): {rho_autolyap:.8f}")
print(f"rho (theory): {rho_theory:.8f}")
print(f"absolute error: {rho_error:.3e}")

if rho_error > rho_tolerance:
    raise AssertionError(
        f"AutoLyap rho differs from the theory bound by more than {rho_tolerance:.1e}."
    )


## Sweeping over the step size $\gamma$

The sweep below uses a small grid so that it is reasonable to run in Colab during
a tutorial. Increase `num_gamma_values` for a smoother plot.


In [ ]:
import matplotlib.pyplot as plt

num_gamma_values = 100

gamma_theory_values = np.linspace(0.0, 2.0 / L, 201)
gamma_values = np.linspace(0.0, 2.0 / L, num_gamma_values + 2)[1:-1]
rho_theory_values = np.maximum(
    np.abs(1.0 - L * gamma_theory_values),
    np.abs(1.0 - mu * gamma_theory_values),
) ** 2

sweep_algorithm = ProximalGradientMethod(gamma=float(gamma_values[0]))
P_sweep, p_sweep, T_sweep, t_sweep = (
    IterationIndependent.LinearConvergence.get_parameters_distance_to_solution(
        sweep_algorithm,
        i=1,
        j=1,
    )
)

rho_autolyap_values = []
for idx, gamma_value in enumerate(gamma_values, start=1):
    sweep_algorithm.set_gamma(float(gamma_value))
    result = IterationIndependent.LinearConvergence.bisection_search_rho(
        problem,
        sweep_algorithm,
        P_sweep,
        T_sweep,
        p=p_sweep,
        t=t_sweep,
        S_equals_T=True,
        s_equals_t=True,
        remove_C3=True,
        lower_bound=0.0,
        upper_bound=1.0,
        tol=1e-4,
        solver_options=solver_options,
        verbosity=0,
    )
    if result["status"] != "feasible":
        raise RuntimeError(
            f"No feasible Lyapunov certificate for gamma={gamma_value:.6f}."
        )
    rho_autolyap_values.append(float(result["rho"]))

    if idx == 1 or idx % 5 == 0 or idx == len(gamma_values):
        print(f"Solved {idx:>2}/{len(gamma_values)}")

rho_autolyap_values = np.array(rho_autolyap_values)

rho_theory_grid_values = np.maximum(
    np.abs(1.0 - L * gamma_values),
    np.abs(1.0 - mu * gamma_values),
) ** 2
max_sweep_error = float(np.max(np.abs(rho_autolyap_values - rho_theory_grid_values)))
print(f"max sweep error: {max_sweep_error:.3e}")
if max_sweep_error > 5e-4:
    raise AssertionError("The sweep differs from the theory curve more than expected.")

fig, ax = plt.subplots(figsize=(6, 3), dpi=160, constrained_layout=True)
ax.plot(
    gamma_theory_values,
    rho_theory_values,
    color="black",
    linewidth=2.0,
    label=r"$\max\{|1-\gamma L|,\ |1-\gamma\mu|\}^2$",
)
ax.scatter(gamma_values, rho_autolyap_values, s=24, color="#1f77b4", label="AutoLyap")
ax.set_xlabel(r"$\gamma$")
ax.set_ylabel(r"$\rho$", rotation=0)
ax.set_title(rf"Proximal gradient: contraction factor vs step size ($L={L:g},\ \mu={mu:g}$)")
ax.set_xlim(0.0, 2.0 / L)
ax.set_ylim(0.3, 1.0)
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()
